# 事前準備：データのダウンロード
以下を実行して、外部ファイルをダウンロードしてください。   
**このセルはColaboratoryを起動するたびに必要となります**

In [ ]:
import urllib.request
!mkdir -p fig sound
urllib.request.urlretrieve('https://park.itc.u-tokyo.ac.jp/yamakata-lab/lecture/mediaproc/mediaproc2/sound/1-Violin-music.wav', './sound/1-Violin-music.wav')

# 音の情報処理1：音の表現（入力・再生・可視化）


第3回で必ず演習してきていただきたいのは以下の５つです。   
ちょっとうんざりするかもしれませんが、実行だけはしてきてください。

- SoundProcessing1.ipynb
- SoundProcessing2.ipynb
- SoundProcessing3.ipynb
- SoundProcessing4.ipynb
- SoundProcessing5.ipynb

課題は授業の当日に同じフォルダで公開します。

## 1  音ファイル
### 1.0 これから処理する音ファイルを確認してください

このノートブックが置かれているフォルダの下に`sound`というフォルダがあり、さらにその下に`1-Violin-music.wav`という音ファイルがあります。   
これは、ヴァイオリンの独奏を収録したものです。   
下記のコードで音を再生することができます。  
どのような音楽か聞いてみてください。

自分のPCで実行している方は、エクスプローラーやFinderなどで当該のファイルをダブルクリックすれば、OSにプリインストールされているサウンドプレイヤ（Windowsであれば"Windows Media Player"など）で再生できます。   


In [ ]:
import IPython
IPython.display.Audio('sound/1-Violin-music.wav')


### 1.1 音ファイルの読み込み
以下のコードで音ファイルをデータとして読み込んでみましょう。  

In [ ]:
import wave
wavfile = wave.open('sound/1-Violin-music.wav', 'rb')

### 1.2 音ファイルのパラメータ
拡張子がwavとなっている場合、これは「RIFF waveform Audio Format」という音声ファイルフォーマットという形式で記録されています。   
このフォーマットは非圧縮です（たとえばmp3などは人間の耳では聞き取れない高周波帯域の音情報を除去している圧縮音響フォーマットです）が、ファイルの冒頭（ヘッダ）に、この音ファイルに関する様々なパラメータが記録されています。  

どのようなパラメータが記録されているかは以下のコードで読み取ることができます。

In [ ]:
# WAVファイルに記録されている情報の表示
print("Channel num : ", wavfile.getnchannels()) # モノラルならば1, ステレオなら2
print("Sample size : ", wavfile.getsampwidth()) # 音声データ1サンプルあたりのバイト数。ビット深度とも呼ぶ。2なら2-byte(16-bit), 3なら3-byte(24-bit)など
print("Sampling rate : ", wavfile.getframerate()) # サンプリングレート。1秒当たり何回データを記録するか
print("Frame num : ", wavfile.getnframes()) # このファイルのフレーム数
print("Sec : ", float(wavfile.getnframes()) / wavfile.getframerate()) # フレーム数/サンプリングレート=録音時間
print("Prams : ", wavfile.getparams()) # このファイルに記録されている各種パラメータを一気に出力

## 2. 音波形の描画

### 2.1 波形のデータ変換とグラフ化

音の波形をグラフとして描画してみましょう。  
この時、横軸を時間にするためにはどうしたらいいでしょうか？   
上で書いた通り、この音声波形が1秒間に何回サンプリングされたかは、`wavfile.getframerate()`で知ることができました。  
よって、x軸は、そのデータのインデックスを` wavfile.getframerate()`で割った値をラベルにしてあげればいいということになりますね。

In [ ]:
%matplotlib inline

import wave
import numpy as np
import matplotlib.pyplot as plt

wavfile = wave.open("sound/1-Violin-music.wav" , "rb" ) # オーディオファイルを開く
# wavfileのデータを最後まで一気に読み込む
#（データサイズが大きいファイルを読み込むときは、読み込むサイズを指定しないとPCがフリーズします！）
data = wavfile.readframes(wavfile.getnframes())
sampling_rate = wavfile.getframerate()  # フレームレート[1/s]
sample_size = wavfile.getsampwidth() # 1サンプルあたりのサイズ
# wavfileをクローズしておく
wavfile.close()

# バイナリデータをint型に変換
data = np.frombuffer(data, dtype= "int16")

# 1サンプルあたりsample_sizeバイトで表されているということは、
# 表現可能な階調は2の(8byte x sample_size)乗
# ただし、波形は正と負があるため、正・負それぞれの領域は、その1/2階調の細かさまで表現できる
# これを最大値として正規化
amp  = 2**(8 * sample_size) / 2
data = data/amp
print('duration: ', len(data))

# X軸をサンプル数から経過時間にするため、X軸の座標を生成
# このデータは0秒からlen(data)/sampling_rate秒まである
# 隣り合うデータとの時間幅は、1.0/sampling_rate秒
x = np.linspace(0, len(data)/sampling_rate, len(data))

# 波形を描画
fig = plt.figure(figsize=(7, 3)) # figure(図を配置する画面)のサイズを指定
ax = plt.subplot() # figureには複数の図を配置できるので、そのうち図を1つ描画する空間を準備

ax.plot(x, data)
ax.set_xlabel("time [sec.]")
ax.set_ylabel("amplitude")
ax.grid()
fig.tight_layout() # 図がはみ出さないようにレイアウト
plt.savefig('fig/SoundProcessing1-1.png') # 図を画像として保存

### 2.2 波形の拡大
先ほどの波形を拡大してみましょう。  
連続的な波のように見える波形は、拡大すると離散的な値のつながりであることが分かります。

In [ ]:
# 波形を描画
start = 1.0 # 開始から1秒後を起点とする
duration = 0.001 # 1ミリ秒分を描画

fig = plt.figure(figsize=(7, 3)) # figure(図を配置する画面)のサイズを指定
ax = plt.subplot() # figureには複数の図を配置できるので、そのうち図を1つ描画する空間を準備

ax.plot(x[round(start*sampling_rate):round(start*sampling_rate+duration*sampling_rate)],
        data[round(start*sampling_rate):round(start*sampling_rate+duration*sampling_rate)], marker="o")

plt.setp(ax.get_xticklabels(), rotation=30, horizontalalignment='right')# 軸を右揃えにして斜めに
ax.grid() # グリッド線を描画
ax.set_xlabel("time [sec.]") # X軸のラベル
ax.set_ylabel("amplitude") # Y軸のラベル
fig.tight_layout() # 図がはみ出さないようにレイアウト
plt.savefig('fig/SoundProcessing1-2.png') # 図を画像として保存